In [1]:
from pathlib import Path
import os
import random

import torch
import torchaudio as ta
import matplotlib.pyplot as plt

root_path = Path.cwd().parent
os.sys.path.insert(0, str(root_path))
import plot_functions as plot
from configs import get_specgram_config
import feature_utils as futils
import utils

In [3]:
def build_mel_transform(cfg: dict, device: torch.device):
    return ta.transforms.MelSpectrogram(
        sample_rate=cfg["sample_rate"],
        n_fft=cfg["n_fft"],
        win_length=cfg["win_length"],
        hop_length=cfg["hop_length"],
        window_fn=cfg["window_fn"],
        power=cfg["power"],
        center=cfg["center"],
        pad_mode=cfg["pad_mode"],
        f_min=cfg["f_min"],
        f_max=cfg["f_max"],
        n_mels=cfg["n_mels"],
        norm="slaney",
        mel_scale="htk",
    ).to(device)


def plot_mel_spectrogram(waveform: torch.Tensor,
                         cfg: dict,
                         mel_tf: ta.transforms.MelSpectrogram,
                         out_path: Path,
                         max_duration_sec: float | None = None):
    """
    waveform: (channels, T)
    """
    sr = cfg["sample_rate"]

    # Optionally crop for faster plotting / smaller PNGs.
    if max_duration_sec is not None:
        max_samples = int(max_duration_sec * sr)
        waveform = waveform[..., :max_samples]

    # If multi-channel, just take first channel for visualization.
    if waveform.dim() == 2 and waveform.size(0) > 1:
        waveform = waveform[:1, :]

    with torch.no_grad():
        mel = mel_tf(waveform)  # (1, n_mels, time_frames)

    # Power -> dB
    mel_db = 10.0 * torch.log10(mel + 1e-10)

    mel_db_np = mel_db.squeeze(0).cpu().numpy()
    n_frames = mel_db_np.shape[1]

    # Time axis in seconds
    t_max = n_frames * cfg["hop_length"] / sr

    # Frequency axis in kHz
    f_max_khz = cfg["f_max"] / 1000.0

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(
        mel_db_np,
        origin="lower",
        aspect="auto",
        extent=[0.0, t_max, 0.0, f_max_khz],
    )
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Frequency [kHz]")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Power [dB]")

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def generate_random_spectrograms(
    data_root: Path,
    out_dir: Path,
    n_samples: int = 1000,
    rng_seed: int = 0,
    max_duration_sec: float | None = None,
):
    """
    data_root: root folder with .wav files (possibly nested).
    out_dir: where to save PNGs.
    """
    cfg = get_specgram_config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    out_dir.mkdir(parents=True, exist_ok=True)

    # Gather all wavs.
    all_wavs = sorted(data_root.rglob("*.wav"))
    if not all_wavs:
        raise FileNotFoundError(f"No .wav files found under {data_root}")

    rng = random.Random(rng_seed)
    selected_files = rng.sample(all_wavs, k=min(n_samples, len(all_wavs)))

    mel_tf = build_mel_transform(cfg, device)

    print(f"Found {len(all_wavs)} wavs, selected {len(selected_files)} for plotting.")

    for i, wav_path in enumerate(selected_files, start=1):
        print(f"[{i}/{len(selected_files)}] {wav_path.name}")

        # Load audio
        waveform, sr = ta.load(str(wav_path))
        if sr != cfg["sample_rate"]:
            waveform = ta.functional.resample(waveform, orig_freq=sr, new_freq=cfg["sample_rate"])

        waveform = waveform.to(device)

        out_path = out_dir / f"{wav_path.stem}_mel.png"
        plot_mel_spectrogram(
            waveform,
            cfg=cfg,
            mel_tf=mel_tf,
            out_path=out_path,
            max_duration_sec=max_duration_sec,
        )

    print("Done.")


if __name__ == "__main__":
    data_root = Path(r"D:\data_dryad")
    out_dir = Path(r"C:\Users\Lindholm\Documents\BSc\bsc_project\research\graphs\manual_wk11\specgrams_30sec")

    generate_random_spectrograms(
        data_root=data_root,
        out_dir=out_dir,
        n_samples=500,
        rng_seed=0,
        max_duration_sec=30,
    )

Found 12095 wavs, selected 500 for plotting.
[1/500] 6230.220903022000.wav
[2/500] 6230.220911032000.wav
[3/500] 6229.220811172000.wav
[4/500] 6229.220930114000.wav
[5/500] 6230.221001184000.wav
[6/500] 6230.220926002000.wav
[7/500] 6230.220907140000.wav
[8/500] 6230.220815110000.wav
[9/500] 6230.220923212000.wav
[10/500] 6230.220827220000.wav
[11/500] 6230.221018044000.wav
[12/500] 6229.220921062000.wav
[13/500] 6230.220930064000.wav
[14/500] 6229.220903044000.wav
[15/500] 6230.220810134000.wav
[16/500] 6229.220903072000.wav
[17/500] 6229.220824020000.wav
[18/500] 6230.221026034000.wav
[19/500] 6229.220928134000.wav
[20/500] 6230.221006150000.wav
[21/500] 6230.221114214000.wav
[22/500] 6230.221022094000.wav
[23/500] 6229.220904224000.wav
[24/500] 6230.220817002000.wav
[25/500] 6229.220824234000.wav
[26/500] 6230.221120122000.wav
[27/500] 6229.220819070000.wav
[28/500] 6230.221110020000.wav
[29/500] 6230.220821134000.wav
[30/500] 6230.220922210000.wav
[31/500] 6230.221012194000.wav
[32